# F3-B — Modelos Clásicos: Random Forest + XGBoost

**Objetivo**: Entrenar Random Forest y XGBoost sobre embeddings + engineered features extraídos en F3-A. Notebook puramente CPU.

**Tiempo estimado**: ~60 min (CPU)


## 1. Instalar dependencias


In [ ]:
!pip install -q polars mlflow xgboost -U


In [ ]:
import polars as pl
import numpy as np
import gc
import os
import json
import time
import mlflow
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
from sklearn.model_selection import ParameterGrid
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
from xgboost import XGBClassifier


## 2. Montar Google Drive y cargar embeddings


In [ ]:
drive.mount('/content/drive')

DRIVE_BASE = "/content/drive/MyDrive/ML/proyecto_integrador"
EMB_DIR = f"{DRIVE_BASE}/embeddings"
REPORTS_DIR = f"{DRIVE_BASE}/reports"
RANDOM_STATE = 42

for d in [REPORTS_DIR]:
    os.makedirs(d, exist_ok=True)

print("Cargando embeddings y features...")
X_train_emb = np.load(f"{EMB_DIR}/train_embeddings.npy")
X_val_emb   = np.load(f"{EMB_DIR}/val_embeddings.npy")
X_test_emb  = np.load(f"{EMB_DIR}/test_embeddings.npy")

eng_train = np.load(f"{EMB_DIR}/train_eng_features.npy")
eng_val   = np.load(f"{EMB_DIR}/val_eng_features.npy")
eng_test  = np.load(f"{EMB_DIR}/test_eng_features.npy")

y_train = np.load(f"{EMB_DIR}/train_labels.npy")
y_val   = np.load(f"{EMB_DIR}/val_labels.npy")
y_test  = np.load(f"{EMB_DIR}/test_labels.npy")

# Concatenar embeddings + engineered features
X_train = np.concatenate([X_train_emb, eng_train], axis=1)
X_val   = np.concatenate([X_val_emb, eng_val], axis=1)
X_test  = np.concatenate([X_test_emb, eng_test], axis=1)

ENG_FEATURE_NAMES = [
    'mayusculas_count', 'char_total', 'exclamacion_count',
    'interrogacion_count', 'porcentaje_mayusculas',
    'puntuacion_emocional', 'total_tokens', 'unique_types', 'ttr'
]

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")


## 3. Helper: evaluación y registro

Función que calcula métricas por clase y globales, y las registra en MLflow.


In [ ]:
results = []

def eval_and_record(name, y_true, y_pred, training_time):
    from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1, 2])
    per_class = {
        label: {'precision': round(p[i], 4), 'recall': round(r[i], 4), 'f1': round(f[i], 4)}
        for i, label in enumerate(['Negativo', 'Neutro', 'Positivo'])
    }
    metrics = {
        'model_name': name,
        'training_time_seconds': round(training_time, 2),
        'f1_macro': round(f1_score(y_true, y_pred, average='macro'), 4),
        'precision_macro': round(precision_score(y_true, y_pred, average='macro'), 4),
        'recall_macro': round(recall_score(y_true, y_pred, average='macro'), 4),
        'accuracy': round(accuracy_score(y_true, y_pred), 4),
        'per_class': per_class,
        'confusion_matrix': confusion_matrix(y_true, y_pred).tolist(),
    }
    results.append(metrics)
    return metrics, y_pred


## 4. Random Forest

Grid search simple sobre `min_samples_leaf` con validación.


In [ ]:
print("=== Random Forest ===")
rf_params = {'n_estimators': [200], 'max_depth': [None], 'min_samples_leaf': [1, 4]}
best_f1 = -1
best_rf = None
best_elapsed = None

for params in ParameterGrid(rf_params):
    start = time.time()
    rf = RandomForestClassifier(**params, random_state=RANDOM_STATE, n_jobs=-1)
    rf.fit(X_train, y_train)
    elapsed = time.time() - start
    val_pred = rf.predict(X_val)
    f1 = f1_score(y_val, val_pred, average='macro')
    print(f"  params={params} -> val_f1={f1:.4f} ({elapsed:.0f}s)")
    if f1 > best_f1:
        best_f1 = f1
        best_rf = rf
        best_elapsed = elapsed

rf = best_rf
y_pred_rf = rf.predict(X_test)
rf_metrics, _ = eval_and_record('Random Forest', y_test, y_pred_rf, best_elapsed)
print(f"RF test F1-macro: {rf_metrics['f1_macro']}")


## 5. Feature importance — Random Forest


In [ ]:
rf_feat_imp = rf.feature_importances_
emb_importance = rf_feat_imp[:768].sum()
eng_importance = rf_feat_imp[768:]

labels_bar = ['Embeddings (768d)'] + ENG_FEATURE_NAMES
values = [emb_importance] + list(eng_importance)

plt.figure(figsize=(10, 6))
colors = ['#3498db'] + ['#2ecc71'] * len(ENG_FEATURE_NAMES)
plt.barh(range(len(values)), values, color=colors, edgecolor='white')
plt.yticks(range(len(values)), labels_bar)
plt.xlabel('Importancia (RF)')
plt.title('Feature Importance: Embeddings vs Engineered (Random Forest)')
plt.tight_layout()
plt.show()


## 6. XGBoost


In [ ]:
print("=== XGBoost ===")
xgb_params = {
    'n_estimators': 300,
    'max_depth': 6,
    'learning_rate': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'eval_metric': 'mlogloss',
    'random_state': RANDOM_STATE,
    'early_stopping_rounds': 10,
}

start = time.time()
xgb = XGBClassifier(**xgb_params)
xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
elapsed = time.time() - start
y_pred_xgb = xgb.predict(X_test)
xgb_metrics, _ = eval_and_record('XGBoost', y_test, y_pred_xgb, elapsed)
print(f"XGBoost test F1-macro: {xgb_metrics['f1_macro']} ({elapsed:.0f}s)")


## 7. Feature importance — XGBoost


In [ ]:
xgb_feat_imp = xgb.feature_importances_
emb_importance = xgb_feat_imp[:768].sum()
eng_importance = xgb_feat_imp[768:]

labels_bar = ['Embeddings (768d)'] + ENG_FEATURE_NAMES
values = [emb_importance] + list(eng_importance)

plt.figure(figsize=(10, 6))
colors = ['#e74c3c'] + ['#f39c12'] * len(ENG_FEATURE_NAMES)
plt.barh(range(len(values)), values, color=colors, edgecolor='white')
plt.yticks(range(len(values)), labels_bar)
plt.xlabel('Importancia (XGBoost)')
plt.title('Feature Importance: Embeddings vs Engineered (XGBoost)')
plt.tight_layout()
plt.show()


## 8. MLflow Tracking


In [ ]:
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "https://humorous-trusting-domelike.ngrok-free.dev")
import requests
try:
    r = requests.get(f"{MLFLOW_TRACKING_URI}/api/2.0/mlflow/experiments/list", timeout=5)
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    print(f"MLflow OK via {MLFLOW_TRACKING_URI}")
except Exception as e:
    print(f"MLflow no disponible: {e}, fallback a SQLite")
    mlflow.set_tracking_uri(f"sqlite:///{DRIVE_BASE}/mlflow_fallback.db")

mlflow.set_experiment("distilbert_clasicos")

for r in results:
    with mlflow.start_run(run_name=r['model_name']):
        mlflow.log_params({'model_name': r['model_name']})
        mlflow.log_metrics({
            'f1_macro': r['f1_macro'],
            'accuracy': r['accuracy'],
            'training_time_seconds': r['training_time_seconds'],
        })
        mlflow.log_dict(r['confusion_matrix'], f"{r['model_name']}_confusion_matrix.json")

print("MLflow tracking completado")


## 9. Exportar métricas a JSON


In [ ]:
report_path = f"{REPORTS_DIR}/metrics_clasicos.json"
with open(report_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f"Exportado: {report_path}")


## 10. Guardar predicciones para Ensemble


In [ ]:
PREDS_DIR = f"{DRIVE_BASE}/preds"
os.makedirs(PREDS_DIR, exist_ok=True)
np.save(f"{PREDS_DIR}/y_pred_rf.npy", y_pred_rf)
np.save(f"{PREDS_DIR}/y_pred_xgb.npy", y_pred_xgb)

part1_results = [r for r in results if r['model_name'] in ['Random Forest', 'XGBoost']]
with open(f"{PREDS_DIR}/part1_results.json", 'w') as f:
    json.dump(part1_results, f, indent=2)
print("Predicciones guardadas para F3-D")


In [ ]:
# Liberar memoria
del X_train, X_val, X_test, X_train_emb, X_val_emb, X_test_emb
del eng_train, eng_val, eng_test
gc.collect()
print("\nF3-B completado. Puede ejecutar F3-D para el ensemble.")
